# 并行实验：成对排序 (Pairwise Ranking)

**评估指标说明：**
根据论文公式 (31)，本实验的核心评估指标是 **系数的 RMSE（即 L2 范数距离）**：
$$ \text{RMSE} = \frac{1}{S} \sum_{s=1}^S \sqrt{\sum_{l=1}^p (\hat{\theta}_{l,s} - \theta_l^*)^2} $$
我们在代码中严格遵循了这一指标。此外，我们保留了“成对准确率 (Pairwise Accuracy)”作为辅助业务指标。

In [ ]:
import os
import json
import time
import numpy as np
%load_ext autoreload
%autoreload 2
import matplotlib.pyplot as plt
from multiprocessing import Pool
from utils.runner import run_single_ranking


In [ ]:
# ==========================================
# 可调参数区 (Hyperparameters)
# ==========================================
NUM_RUNS = 200       # 重复实验次数
NUM_WORKERS = 10     # 并行进程数

params = {
    'm': 10,         # 节点数量
    'n': 100,        # 每个节点的样本量
    'p': 5,          # 特征维度 (论文中 Ranking 实验 p=5)
    'p_prime': 5,    # 非零特征数量
    'pc': 0.3,       # 网络连接概率
    'T': 20,         # 外层 ADMM 迭代次数 (控制全局收敛，可调 10-50)
    'W_inner': 5,    # 内层 ADMM 迭代次数 (控制本地共识，可调 1-10)
    'rho': 0.13,     # ADMM 惩罚参数 (核心参数！控制共识强度，太大收敛慢，太小不共识)
    'lam_t': 0.0,    # L1 正则化系数 (如果真实参数不稀疏，设为 0)
    'noise_type': 'normal', # 噪声类型: 'normal', 'exp', 'cauchy', 't1', 't3'
    
    # 控制要跑哪些算法 (True 表示运行，False 表示跳过)
    'run_proposed': True,
    'run_local': True,
    'run_pooled': False, # Pooled 比较慢，可以选择性开启
    'run_dgd': True
}

os.makedirs('ranking', exist_ok=True)
filename = f"ranking/exp1_ranking_{NUM_RUNS}_{params['noise_type']}_p{params['p']}_pc_{str(params['pc']).replace('.', '')}_rho_{str(params['rho']).replace('.', '')}.json"

print(f"Starting Parallel Ranking Experiments: {NUM_RUNS} runs...")
results = []
def update_progress(result):
    results.append(result)
    print(f"\rProgress: {len(results)}/{NUM_RUNS} runs completed.", end="", flush=True)

def log_error(e):
    print(f"\nError in worker process: {e}")

with Pool(NUM_WORKERS) as pool:
    for i in range(NUM_RUNS):
        pool.apply_async(run_single_ranking, args=(i, params), callback=update_progress, error_callback=log_error)
    pool.close()
    pool.join()
print() # 换行

# 保存结果
with open(filename, 'w', encoding='utf-8') as f:
    json.dump({'parameters': params, 'results': results}, f, ensure_ascii=False, indent=4)
print(f"Results saved to {filename}")

In [ ]:
# ==========================================
# 实验完成
# ==========================================
print(f"\n实验完成！数据已保存至 {filename}")
print("请打开 exp1_plot_ranking.ipynb 读取该文件以生成 Markdown 表格和收敛曲线。\n")
